In [1]:
# -*- coding: utf-8 -*-
"""
CTR Baseline — DCN-V2 + DIN(MLP-attention) [Notebook, OOM-safe, NaN-safe]
- 시퀀스: 해시 임베딩(고정 버킷), Lazy 파싱 → 대용량 안전
- 본체: Deep & Cross v2(CrossNetMix: low-rank + experts) + Deep MLP
- 시퀀스 분기: DIN 정석(activation unit MLP)로 타깃×히스토리 로컬 어텐션
- 범주형: train 기준 factorize(OOV), 연속형: BatchNorm1d
- 불균형: pos_weight, 지표: AUC/PR-AUC + EarlyStopping
- 안정성: fp16 마스킹 오버플로 방지, 전부 PAD 시퀀스 안전 처리, NaN/±inf 정리
"""

import os, re, json, random, gc, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# =============== Utils ===============
def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"]=str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def infer_device(arg="auto"):
    if arg=="cpu": return torch.device("cpu")
    if arg=="cuda" or (arg=="auto" and torch.cuda.is_available()):
        print("Using CUDA"); return torch.device("cuda")
    print("Using CPU"); return torch.device("cpu")

def parse_seq_string(s: str):
    if s is None: return []
    s = str(s).strip()
    if not s or s.lower()=="nan": return []
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]; toks = [t.strip().strip("'\"") for t in s.split(",")]
    else:
        s = re.sub(r"[^\d]+", ",", s); toks = [t for t in s.split(",") if t]
    out=[]
    for t in toks:
        try: out.append(int(t))
        except Exception:
            try: out.append(int(float(t)))
            except Exception: pass
    return out

def emb_dim_from_card(card, cap=64):
    return int(min(cap, max(8, round(1.6*(card**0.25)))))

# =============== Config ===============
class Args:
    train="./train.parquet"; test="./test.parquet"
    label="clicked"; seq_col="seq"; id_col="ID"

    sample_subset=1.0            # 파이프라인 검증은 0.02~0.1 권장
    max_seq_len=50
    HASH_BUCKETS=262_144         # 2^18; 여유되면 524_288
    seq_emb_dim=64
    dropout=0.2

    # DCN-V2 (CrossNetMix)
    cross_layers=3
    cross_low_rank=32
    cross_num_experts=4

    # Deep tower
    deep_units=[512,256,128]

    # Train
    epochs=3; batch_size=512; lr=1e-3; patience=2
    seed=42; device="auto"; num_workers=0; pin_memory=False
args=Args()
set_seed(args.seed); device=infer_device(args.device)

# =============== (옵션) 더미 데이터 생성 (파일 없을 때만) ===============
def create_dummy(num_rows, is_test=False):
    rng=np.random.default_rng(0)
    df=pd.DataFrame({
        "cont_feat1": rng.normal(size=num_rows),
        "cont_feat2": rng.random(num_rows)*10,
        "gender": rng.choice(["M","F","U"], num_rows, p=[0.45,0.45,0.1]),
        "age_group": rng.choice(["10s","20s","30s","40s+"], num_rows),
        "inventory_id": rng.integers(500, 520, num_rows),
        "day_of_week": rng.integers(0,7,num_rows),
        "hour": rng.integers(0,24,num_rows),
        "l_feat_14": rng.integers(1000,1050,num_rows),
    })
    seqs=[]
    for _ in range(num_rows):
        fmt=rng.choice(["pipe","comma","json","empty","single"])
        L=int(rng.integers(1,args.max_seq_len+10))
        items=[str(int(rng.integers(1000,1500))) for _ in range(L)]
        if fmt=="pipe": seqs.append("|".join(items))
        elif fmt=="comma": seqs.append(",".join(items))
        elif fmt=="json": seqs.append(f"[{','.join(items)}]")
        elif fmt=="empty": seqs.append(np.nan)
        else: seqs.append(items[0])
    df[args.seq_col]=seqs
    if is_test: df[args.id_col]=np.arange(num_rows)
    else: df[args.label]=rng.integers(0,2,num_rows)
    return df

if not os.path.exists(args.train):
    print("train.parquet 없음 → 더미 20k 생성"); create_dummy(20_000,False).to_parquet(args.train)
if not os.path.exists(args.test):
    print("test.parquet 없음 → 더미 5k 생성"); create_dummy(5_000,True).to_parquet(args.test)

# =============== Load & Prep ===============
print("[1/7] Load parquet ...")
train=pd.read_parquet(args.train); test=pd.read_parquet(args.test)
assert args.label in train.columns and args.seq_col in train.columns

if 0<args.sample_subset<1.0:
    print(f"Subsampling to {args.sample_subset*100:.1f}%")
    train=train.sample(frac=args.sample_subset, random_state=args.seed).reset_index(drop=True)

must_drop={args.label,args.seq_col,args.id_col}
base_cols=[c for c in train.columns if c not in must_drop]

# ---- 타깃 자동 선택 + 범주형/연속형 분리 ----
fixed_cats = [c for c in ["gender","age_group","inventory_id","day_of_week","hour"] if c in base_cols]
all_lfeats = [c for c in base_cols if c.startswith("l_feat_")]

preferred_targets = ["ad_id","creative_id","l_feat_14"]   # 우선순위(존재하면 사용)
present_targets = [c for c in preferred_targets if c in base_cols]
target_name = present_targets[0] if len(present_targets)>0 else None
if target_name is None and "l_feat_14" in all_lfeats:
    target_name = "l_feat_14"

CAT_CARD_MAX = 200_000  # 임베딩 메모리 보호
cand_cats = fixed_cats.copy()

# 임시 split로 카디널리티 대략 파악
tr_tmp, va_tmp = train_test_split(train, test_size=0.15, random_state=args.seed, stratify=train[args.label])
for c in all_lfeats:
    try:
        nunq = max(tr_tmp[c].nunique(dropna=True), va_tmp[c].nunique(dropna=True))
    except Exception:
        nunq = pd.concat([tr_tmp[c], va_tmp[c]], axis=0).nunique(dropna=True)
    if nunq <= CAT_CARD_MAX:
        cand_cats.append(c)

# 타깃은 반드시 포함
if target_name is not None and target_name not in cand_cats and target_name in base_cols:
    cand_cats.append(target_name)

cont_cols=[c for c in base_cols if c not in cand_cats]
print(f"[1.1] target_name={target_name}  cont={len(cont_cols)}  cat={len(cand_cats)}")

# 최종 split
tr,va=train_test_split(train, test_size=0.15, random_state=args.seed, stratify=train[args.label]); del train; gc.collect()

# 범주형 맵 (train 기준)
cat_maps, cat_cards = {}, {}
for c in cand_cats:
    cats=pd.Categorical(tr[c])
    cat_maps[c]={v:i for i,v in enumerate(cats.categories)}
    cat_cards[c]=len(cats.categories)

# 시퀀스 해시 임베딩
PAD_ID, OOF_ID, SEQ_BASE = 0, 1, 2
seq_vocab_size = args.HASH_BUCKETS + SEQ_BASE
def seq_to_ids_hash(lst, max_len=50):
    ids=[SEQ_BASE + (int(t)%args.HASH_BUCKETS) for t in lst][-max_len:]
    if len(ids)<max_len: ids=[PAD_ID]*(max_len-len(ids))+ids
    return np.array(ids, dtype=np.int32)

# =============== Dataset (Lazy seq parsing) ===============
class CTRDataset(Dataset):
    def __init__(self, df, cont_cols, cat_cols, cat_maps, seq_col, max_seq_len, label_col=None):
        self.df=df.reset_index(drop=True)
        self.cont_cols, self.cat_cols = cont_cols, cat_cols
        self.cat_maps, self.seq_col = cat_maps, seq_col
        self.max_seq_len=max_seq_len; self.has_label=label_col is not None; self.label_col=label_col
        self.Xc=self.df[self.cont_cols].astype(np.float32).fillna(0.0).values if self.cont_cols else None
        self.Xcats={c:self.df[c].map(self.cat_maps[c]).fillna(cat_cards[c]).astype(np.int64).values for c in self.cat_cols}
        if self.has_label: self.y=self.df[self.label_col].astype(np.float32).values
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        xc=torch.from_numpy(self.Xc[idx]) if self.Xc is not None else torch.empty(0)
        cats={c: torch.tensor(self.Xcats[c][idx],dtype=torch.long) for c in self.cat_cols}
        lst=parse_seq_string(self.df.at[idx,self.seq_col])
        seq=torch.from_numpy(seq_to_ids_hash(lst, self.max_seq_len)).long()
        if self.has_label: return xc, cats, seq, torch.tensor(self.y[idx],dtype=torch.float32)
        return xc, cats, seq

# =============== DCN-V2 (CrossNetMix) ===============
class CrossLayerMix(nn.Module):
    """CrossNetMix layer (low-rank + experts, 간소화)"""
    def __init__(self, dim, low_rank=32, num_experts=4):
        super().__init__()
        self.num_experts=num_experts
        self.U=nn.Parameter(torch.randn(num_experts, dim, low_rank)*0.01)
        self.V=nn.Parameter(torch.randn(num_experts, low_rank, dim)*0.01)
        self.C=nn.Parameter(torch.zeros(num_experts, dim))
        self.gating=nn.Linear(dim, num_experts, bias=False)
        self.bias=nn.Parameter(torch.zeros(dim))
    def forward(self, x0, x):
        gate=torch.softmax(self.gating(x), dim=-1)          # (B,E)
        Xu=torch.einsum("bd,edr->ber", x, self.U)           # (B,E,r)
        Xuv=torch.einsum("ber,erd->bed", Xu, self.V)        # (B,E,d)
        Xuv = Xuv + self.C                                  # (B,E,d)
        mix=torch.einsum("be,bed->bd", gate, Xuv)           # (B,d)
        out=x0*mix + x + self.bias                          # cross + residual
        return out

class CrossNetMix(nn.Module):
    def __init__(self, dim, num_layers=3, low_rank=32, num_experts=4):
        super().__init__()
        self.layers=nn.ModuleList([CrossLayerMix(dim, low_rank, num_experts) for _ in range(num_layers)])
    def forward(self, x):
        x0=x; xl=x
        for layer in self.layers:
            xl=layer(x0, xl)
        return xl

# =============== DIN(정석) Activation Unit ===============
class DINActivationUnit(nn.Module):
    """w_i = MLP([q, k_i, q-k_i, q*k_i])"""
    def __init__(self, dim_q, dim_k, hidden=[64,32], dropout=0.0):
        super().__init__()
        in_dim = dim_q + dim_k + dim_q + dim_k
        layers=[]
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim=h
        layers += [nn.Linear(in_dim, 1)]
        self.net=nn.Sequential(*layers)
    def forward(self, q, K):
        # q: (B,Dq), K: (B,L,Dk)
        B,L,Dk = K.shape
        q_exp = q.unsqueeze(1).expand(-1,L,-1)        # (B,L,Dq)
        feats = [q_exp, K, q_exp-K, q_exp*K]
        z = torch.cat(feats, dim=2)                   # (B,L,·)
        w = self.net(z).squeeze(2)                    # (B,L)
        return w

# =============== Model: DCN-V2 + DIN(MLP-attn) ===============
class DCN_DIN_Model(nn.Module):
    def __init__(self, cont_dim, cat_cards, seq_vocab_size, target_name=None,
                 seq_emb_dim=64, deep_units=[512,256,128],
                 cross_layers=3, cross_low_rank=32, cross_num_experts=4, dropout=0.2):
        super().__init__()
        self.has_cont=cont_dim>0
        if self.has_cont: self.bn=nn.BatchNorm1d(cont_dim)

        # categorical embeddings
        self.cat_embs=nn.ModuleDict()
        for name, card in cat_cards.items():
            dim=emb_dim_from_card(card+1)
            self.cat_embs[name]=nn.Embedding(card+1+1, dim)  # +1 OOV
        self.cat_total_dim = sum(emb.embedding_dim for emb in self.cat_embs.values())

        # base tabular x0
        self.tab_dim = (cont_dim if self.has_cont else 0) + self.cat_total_dim

        # DCN-V2
        self.cross = CrossNetMix(self.tab_dim, num_layers=cross_layers,
                                 low_rank=cross_low_rank, num_experts=cross_num_experts)

        # Deep tower
        deep_layers=[]; in_dim=self.tab_dim
        for h in deep_units:
            deep_layers += [nn.Linear(in_dim,h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim=h
        self.deep = nn.Sequential(*deep_layers)

        # Sequence + DIN(activation unit)
        self.seq_emb = nn.Embedding(seq_vocab_size, seq_emb_dim, padding_idx=PAD_ID)
        self.target_name = target_name if (target_name in self.cat_embs) else None
        if self.target_name is not None:
            tdim = self.cat_embs[self.target_name].embedding_dim
        else:
            tdim = seq_emb_dim
        self.proj_t = nn.Linear(tdim, seq_emb_dim, bias=False) if tdim!=seq_emb_dim else nn.Identity()
        self.din_act = DINActivationUnit(seq_emb_dim, seq_emb_dim, hidden=[64,32], dropout=dropout)

        # Final head: [cross_out, deep_out, interest_vec]
        final_in = self.tab_dim + deep_units[-1] + seq_emb_dim
        self.head = nn.Sequential(
            nn.Linear(final_in, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def forward(self, xc, xcats, seq_ids):
        # ----- Tabular -----
        outs=[]
        if self.has_cont: outs.append(self.bn(xc))
        if len(self.cat_embs)>0:
            outs.append(torch.cat([self.cat_embs[n](xcats[n]) for n in self.cat_embs], dim=1))
        x0 = torch.cat(outs, dim=1) if len(outs)>1 else outs[0]   # (B, tab_dim)

        # DCN-V2 cross / Deep tower
        x_cross = self.cross(x0)                                  # (B, tab_dim)
        x_deep  = self.deep(x0)                                   # (B, deep_dim)

        # ----- DIN -----
        K = self.seq_emb(seq_ids)                                 # (B,L,D)
        if self.target_name is not None:
            q = self.cat_embs[self.target_name](xcats[self.target_name])  # (B,Dt)
            q = self.proj_t(q)                                    # (B,D)
        else:
            mask=(seq_ids!=PAD_ID).float().unsqueeze(-1)
            q = (K*mask).sum(dim=1)/(mask.sum(dim=1)+1e-8)        # mean fallback

        # Activation unit -> attention logits
        logits_w = self.din_act(q, K)                             # (B,L)
        maskL = (seq_ids != PAD_ID)

        # fp16에서도 안전한 음수값으로 PAD 마스킹
        neg = torch.finfo(logits_w.dtype).min
        logits_w = logits_w.masked_fill(~maskL, neg)

        # 전부 PAD인 행 안전 처리 (alpha=0)
        valid = maskL.any(dim=1, keepdim=True)                    # (B,1)
        maxv = torch.where(valid,
                           logits_w.max(dim=1, keepdim=True).values,
                           torch.zeros_like(logits_w[:, :1]))
        logits_w = torch.where(valid, logits_w - maxv, torch.zeros_like(logits_w))
        alpha = torch.where(valid, torch.softmax(logits_w, dim=1), torch.zeros_like(logits_w))
        alpha = torch.nan_to_num(alpha, nan=0.0)

        interest = torch.sum(K * alpha.unsqueeze(2), dim=1)       # (B,D)

        # 출력 결합
        z = torch.cat([x_cross, x_deep, interest], dim=1)
        return self.head(z).squeeze(1)

# =============== Data & Loaders ===============
ds_tr=CTRDataset(tr, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=args.label)
ds_va=CTRDataset(va, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=args.label)
ds_te=CTRDataset(test, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=None)

n_pos=float((tr[args.label]==1).sum()); n_neg=float(len(tr)-n_pos)
pos_weight=torch.tensor(max(1.0, n_neg/max(1.0,n_pos)), dtype=torch.float32, device=device)
print(f"[2/7] Train label: pos={int(n_pos)} neg={int(n_neg)}  pos_weight={pos_weight.item():.3f}")

dl_tr=DataLoader(ds_tr, batch_size=args.batch_size, shuffle=True,
                 num_workers=args.num_workers, pin_memory=args.pin_memory)
dl_va=DataLoader(ds_va, batch_size=args.batch_size, shuffle=False,
                 num_workers=args.num_workers, pin_memory=args.pin_memory)
dl_te=DataLoader(ds_te, batch_size=args.batch_size, shuffle=False,
                 num_workers=args.num_workers, pin_memory=args.pin_memory)

# =============== Train ===============
print("[3/7] Build model ...")
model=DCN_DIN_Model(
    cont_dim=len(cont_cols), cat_cards=cat_cards, seq_vocab_size=seq_vocab_size,
    target_name=target_name, seq_emb_dim=args.seq_emb_dim,
    deep_units=args.deep_units, cross_layers=args.cross_layers,
    cross_low_rank=args.cross_low_rank, cross_num_experts=args.cross_num_experts,
    dropout=args.dropout
).to(device)

criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer=torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
scaler=GradScaler(enabled=(device.type=="cuda"))

best_auc, best_state, wait=-1.0, None, 0
final_epoch, final_prauc=0, 0.0

print("[4/7] Train ...")
for epoch in range(1, args.epochs+1):
    model.train(); tr_loss=0.0
    for xc, cats, seq, y in dl_tr:
        xc, seq, y = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True), y.to(device, non_blocking=True)
        cats_dev={k:v.to(device, non_blocking=True) for k,v in cats.items()}
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device.type=="cuda")):
            logits=model(xc, cats_dev, seq)
            loss=criterion(logits, y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        tr_loss += loss.item()*y.size(0)
    tr_loss/=len(ds_tr); scheduler.step()

    # ----- Validation -----
    model.eval(); va_loss=0.0; ys=[]; ps=[]
    with torch.no_grad():
        for xc, cats, seq, y in dl_va:
            xc, seq = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True)
            cats_dev={k:v.to(device, non_blocking=True) for k,v in cats.items()}
            with autocast(enabled=(device.type=="cuda")):
                logits=model(xc, cats_dev, seq)
                loss=criterion(logits, y.to(device))
                prob=torch.sigmoid(logits)
            # NaN/±inf 정리
            p = prob.detach().cpu().numpy().astype(np.float64)
            p = np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0)
            va_loss += loss.item()*len(y)
            ys.append(y.numpy()); ps.append(p)
    va_loss/=len(ds_va)
    y_true=np.concatenate(ys); y_prob=np.concatenate(ps)
    auc=roc_auc_score(y_true, y_prob); prauc=average_precision_score(y_true, y_prob)
    print(f"Epoch {epoch:02d} | Train {tr_loss:.5f} | Val {va_loss:.5f} | AUC {auc:.6f} | PR-AUC {prauc:.6f}")

    if auc > best_auc + 1e-5:
        best_auc, final_prauc, final_epoch = auc, prauc, epoch
        best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
    else:
        wait += 1
        if wait >= args.patience:
            print(f"Early stopping. Best AUC={best_auc:.6f} at epoch {final_epoch}."); break

if best_state is not None:
    model.load_state_dict(best_state)

# =============== Inference ===============
print("[5/7] Predict test ...")
model.eval(); probs=[]
with torch.no_grad():
    for xc, cats, seq in dl_te:
        xc, seq = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True)
        cats_dev={k:v.to(device, non_blocking=True) for k,v in cats.items()}
        with autocast(enabled=(device.type=="cuda")):
            logits=model(xc, cats_dev, seq)
            prob=torch.sigmoid(logits)
        p = prob.detach().cpu().numpy().astype(np.float64)
        p = np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0)
        probs.append(p)
probs=np.concatenate(probs)

# =============== Save ===============
print("[6/7] Save outputs ...]")
sub_path="./din_dcnv2_submit.csv"; meta_path="./din_dcnv2_meta.json"
pd.DataFrame({args.id_col: test[args.id_col].values, "clicked": probs}).to_csv(sub_path, index=False)
meta={
    "columns":{"continuous":cont_cols, "categorical":cand_cats},
    "seq_vocab":{"type":"hash","buckets":args.HASH_BUCKETS,"pad_id":0,"oov_id":1},
    "hyperparameters":{
        "sample_subset":args.sample_subset,"max_seq_len":args.max_seq_len,
        "seq_emb_dim":args.seq_emb_dim,"dropout":args.dropout,
        "epochs":args.epochs,"batch_size":args.batch_size,"lr":args.lr,
        "cross_layers":args.cross_layers,"cross_low_rank":args.cross_low_rank,"cross_num_experts":args.cross_num_experts,
        "deep_units":args.deep_units
    },
    "performance":{"best_epoch":int(final_epoch),"AUC":float(best_auc),"PR_AUC":float(final_prauc)},
    "target_name": target_name
}
with open(meta_path,"w",encoding="utf-8") as f: json.dump(meta,f,ensure_ascii=False,indent=2)
print(f"[7/7] Done. \n - {sub_path}\n - {meta_path}")


Using CUDA
[1/7] Load parquet ...
[1.1] target_name=l_feat_14  cont=85  cat=32
[2/7] Train label: pos=173552 neg=8925000  pos_weight=51.426
[3/7] Build model ...
[4/7] Train ...
Epoch 01 | Train 1.21088 | Val nan | AUC 0.734113 | PR-AUC 0.065231
Epoch 02 | Train 1.18769 | Val nan | AUC 0.737506 | PR-AUC 0.074199
Epoch 03 | Train 1.16703 | Val nan | AUC 0.740209 | PR-AUC 0.077473
[5/7] Predict test ...
[6/7] Save outputs ...]
[7/7] Done. ⬇️
 - ./din_dcnv2_submit.csv
 - ./din_dcnv2_meta.json
